In [ ]:
sampling_interval = '1h'

In [ ]:
data_path = "/home/lkapral/hb/data/"

In [ ]:
import numpy as np
import pandas as pd
import tqdm
import argparse
import os
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

tqdm.tqdm.pandas()

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
bp = pd.read_parquet(os.path.join(data_path, 'blood-pressure-1min-pivot-20240807.parquet'))

In [ ]:
blood = pd.read_parquet(os.path.join(data_path, 'blood-products-20260630.parquet'))


In [ ]:
for col in blood.columns:
   print(blood[col].value_counts()) 
   print(blood[col].describe()) 

In [ ]:
cate = pd.read_parquet(os.path.join(data_path,  'catecholamines-1min-pivot-20240719.parquet'))

In [ ]:
vitals = pd.read_parquet(os.path.join(data_path,  'vital-signs-20250502.parquet'))

In [ ]:
demographics = pd.read_parquet(os.path.join(data_path,  'demographic-1min-pivot-20240719.parquet'))

In [ ]:
intake = pd.read_parquet(os.path.join(data_path,  'intake-1min-pivot-20240719.parquet'))

In [ ]:
labs = pd.read_parquet(os.path.join(data_path, 'lab-results-1min-pivot-20241008.parquet'))

In [ ]:
sitecare = pd.read_parquet(os.path.join(data_path,  'sitecare-1min-pivot-20240719.parquet'))

In [ ]:
bp = bp.reset_index(drop=False)

In [ ]:
demographics

In [ ]:
labs_re = pd.read_parquet(os.path.join(data_path, 'resampled/lab-results-1min-pivot-20241008_resampled.parquet'))

In [ ]:
labs_re['hemoglobin_g/dl'].value_counts()

In [ ]:
import pandas as pd

# 1) Make sure `label` is a category (so comparisons and filtering are faster)
vitals["label"] = vitals["label"].astype("category")

# 2) Build a Series for heart_rate:
hr = (
    vitals[vitals["label"] == "heart_rate"]
    .loc[:, ["encounterId", "utcChartTime", "valueNumber"]]
    .drop_duplicates(subset=["encounterId", "utcChartTime"], keep="first")
    .set_index(["encounterId", "utcChartTime"])["valueNumber"]
)
hr.name = "heart_rate"

# 3) Build a Series for respiratory_rate:
rr = (
    vitals[vitals["label"] == "respiratory_rate"]
    .loc[:, ["encounterId", "utcChartTime", "valueNumber"]]
    .drop_duplicates(subset=["encounterId", "utcChartTime"], keep="first")
    .set_index(["encounterId", "utcChartTime"])["valueNumber"]
)
rr.name = "respiratory_rate"

# 4) Build a Series for spO2 (if any):
sp = (
    vitals[vitals["label"] == "spO2"]
    .loc[:, ["encounterId", "utcChartTime", "valueNumber"]]
    .drop_duplicates(subset=["encounterId", "utcChartTime"], keep="first")
    .set_index(["encounterId", "utcChartTime"])["valueNumber"]
)
sp.name = "spO2"

# 5) Concatenate the three Series along axis=1
vitals_wide = pd.concat([hr, rr], axis=1).reset_index()


In [ ]:
vitals_wide.to_parquet(os.path.join(data_path, 'vitals_wide.parquet'))

In [ ]:
vitals_wide['heart_rate'].describe()

In [ ]:
# No resampling for demographics
demographics.drop(columns=['utcChartTime'], inplace=True)

In [ ]:
# Convert encounterId to integer and drop rows where this conversion is not possible
dataframes = [bp, cate, intake, labs, sitecare, demographics, vitals_wide]

for df in dataframes:
    df['encounterId'] = pd.to_numeric(df['encounterId'], errors='coerce')
    df.dropna(subset=['encounterId'], inplace=True)
    df['encounterId'] = df['encounterId'].astype(int)

In [ ]:
sitecare[sitecare['encounterId']=='nan']

In [ ]:
vitals_wide

In [ ]:
bp.loc[bp['encounterId']==383478,['utcChartTime','blood_pressure_systolic_mmHg']].head(20)

In [ ]:
import pandas as pd
import numpy as np

def downcast_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Downcast floats and ints where possible, and convert object/string columns
    to pandas.Categorical (if they have few unique values). Returns a new df.
    """
    df2 = df.copy()

    for col in df2.columns:
        col_data = df2[col]

        if pd.api.types.is_float_dtype(col_data.dtype):
            df2[col] = pd.to_numeric(col_data, downcast='float')

        elif pd.api.types.is_integer_dtype(col_data.dtype):
            df2[col] = pd.to_numeric(col_data, downcast='integer')

        elif pd.api.types.is_object_dtype(col_data.dtype):
            num_unique = col_data.nunique(dropna=False)
            num_total  = len(col_data)
            # only convert to categorical if “small enough”
            if 0 < num_unique < num_total / 2:
                df2[col] = col_data.astype('category')

        # If it’s already a datetime, leave it alone.
        # If it’s some other custom type, you can check & downcast here as needed.

    return df2


In [ ]:
bp       = downcast_df(bp)
cate     = downcast_df(cate)
intake   = downcast_df(intake)
labs     = downcast_df(labs)
sitecare = downcast_df(sitecare)
demographics = downcast_df(demographics)
vitals_wide = downcast_df(vitals_wide)

In [ ]:
def fast_resample_and_reindex(
    df: pd.DataFrame,
    time_column: str,
    interval: str = '1h',
    full_time_index: pd.DatetimeIndex = None,
    const_cols: list[str] = None,
) -> pd.DataFrame:
    """
    1) Floors the timestamp to the given interval.
    2) Groups by (encounterId, time_bin) using pd.Grouper, then aggregates.
    3) Reindexes to a full MultiIndex if full_time_index is provided.
    4) Fills 'sum' columns with 0 and 'mean/min' columns by ffill/bfill using transform().
    5) Merges constant‐per‐encounter columns (e.g. age, sex) at the end.

    - interval: pandas offset alias, e.g. '1h' or '30T'
    - full_time_index: if provided, must be a DatetimeIndex of all possible time‐bins
    - const_cols: list of columns that are constant per encounter (e.g. ['age', 'sex'])
    """
    # 1) Copy & floor timestamps
    df2 = df.copy()
    df2[time_column] = pd.to_datetime(df2[time_column]).dt.floor(interval)

    if 'encounterId' not in df2.columns:
        raise ValueError("DataFrame must contain 'encounterId'")

    # 2) Extract constant columns (so we don't have to recompute mode() per‐interval)
    const_cols = const_cols or []
    const_cols = [c for c in const_cols if c in df2.columns]
    if const_cols:
        const_df = (
            df2
            .loc[:, ['encounterId'] + const_cols]
            .drop_duplicates(subset=['encounterId'])
            .set_index('encounterId')
        )
    else:
        const_df = None

    # 3) Build the per‐column aggregation spec (only keep cols actually in df2)
    agg_spec: dict[str, str] = {
        'blood_pressure_diastolic_mmHg': 'mean',
        'blood_pressure_mean_mmHg':      'mean',
        'blood_pressure_systolic_mmHg':  'mean',
        'dobutamine_µg/kg/min':          'mean',
        'fibrinogen_mg/dl':              'mean',
        'lactate_mmol/l':                'mean',
        'norepinephrine_µg/kg/min':      'mean',
        'platelet_count_G/l':            'mean',
        'vasopressin_IE/h':              'mean',
        'colloids_ml':                   'sum',
        'easy-flow_ml':                  'sum',
        'fluids_ml':                     'sum',
        'harnk_ml':                      'sum',
        'jackson-pratt_ml':              'sum',
        'redon-drain_ml':                'sum',
        'robinson-drain_ml':             'sum',
        'thorax-drain_ml':               'sum',
        'hemoglobin_g/dl':               'min',
        'base_excess_mmol/l':            'min',
        'platelet_count_G/l':            'min',
        'heartrate':                     'mean',
        'respiratory_rate':              'mean',
        # (Do not include age/sex here if they’re in const_cols.)
    }
    valid_agg = {col: op for col, op in agg_spec.items() if col in df2.columns}

    # 4) One‐shot grouping via pd.Grouper
    grouped = (
        df2
        .groupby(
            ['encounterId', pd.Grouper(key=time_column, freq=interval)],
            observed=True
        )
        .agg(valid_agg)
    )
    grouped.index.rename(['encounterId', time_column], inplace=True)

    # 5) Reindex to the full MultiIndex if requested
    if full_time_index is not None:
        encounters = grouped.index.get_level_values('encounterId').unique()
        full_mi = pd.MultiIndex.from_product(
            [encounters, full_time_index],
            names=['encounterId', time_column]
        )
        grouped = grouped.reindex(full_mi)

    # 6) Fill missing values
    sums   = [c for c, op in valid_agg.items() if op == 'sum']
    meanmin = [c for c, op in valid_agg.items() if op in ('mean', 'min')]

    if sums:
        grouped[sums] = grouped[sums].fillna(0)

    if meanmin:
        # Use transform() instead of apply() so the index stays (encounterId,time)
        grouped[meanmin] = (
            grouped
            .groupby(level='encounterId')[meanmin]
            .transform(lambda grp: grp.ffill().bfill())
        )

    # 7) Reset index back to columns
    result = grouped.reset_index()

    # 8) Merge constant columns back in
    if const_df is not None:
        result = result.merge(
            const_df.reset_index(),
            on='encounterId',
            how='left'
        )

    return result



sampling_interval = '1h'

# 1) Build full_time_index once
all_times = pd.concat([
    bp['utcChartTime'],
    cate['utcChartTime'],
    intake['utcChartTime'],
    labs['utcChartTime'],
    sitecare['utcChartTime'],
    vitals_wide['utcChartTime']
]).dt.floor(sampling_interval)

full_time_index = pd.date_range(
    start=all_times.min(),
    end=all_times.max(),
    freq=sampling_interval
)


vitals_fast = fast_resample_and_reindex(
    vitals_wide,
    time_column='utcChartTime',
    interval=sampling_interval,
    full_time_index=full_time_index,
    const_cols=[]
)
del vitals_wide

# 2) Resample each table, watching the printed steps
bp_fast = fast_resample_and_reindex(
    bp,
    time_column='utcChartTime',
    interval=sampling_interval,
    full_time_index=full_time_index,
    const_cols=['age', 'sex_or_gender_numeric']
)
del bp

cate_fast = fast_resample_and_reindex(
    cate,
    time_column='utcChartTime',
    interval=sampling_interval,
    full_time_index=full_time_index,
    const_cols=[]
)
del cate
intake_fast = fast_resample_and_reindex(
    intake,
    time_column='utcChartTime',
    interval=sampling_interval,
    full_time_index=full_time_index,
    const_cols=[]
)
del intake
labs_fast = fast_resample_and_reindex(
    labs,
    time_column='utcChartTime',
    interval=sampling_interval,
    full_time_index=full_time_index,
    const_cols=[]
)
del labs
sitecare_fast = fast_resample_and_reindex(
    sitecare,
    time_column='utcChartTime',
    interval=sampling_interval,
    full_time_index=full_time_index,
    const_cols=[]
)
del sitecare





# 3) Merge them all—and watch prints to verify each DataFrame was built
merged_df = bp_fast
for df in [cate_fast, intake_fast, labs_fast, sitecare_fast, vitals_fast]:
    print("Merging another table into merged_df; current shape:", merged_df.shape)
    merged_df = merged_df.merge(
        df,
        on=['encounterId', 'utcChartTime'],
        how='outer'
    )
    print("   → New merged_df shape:", merged_df.shape)

# 4) Finally bring in demographics
print("Merging demographics; merged_df shape before:", merged_df.shape)
merged_df = merged_df.merge(demographics, on='encounterId', how='left')
print("Final merged_df shape:", merged_df.shape)

print("Done! Your merged DataFrame has", merged_df.shape, "rows ×", merged_df.shape[1], "columns.")

In [ ]:
sampling_interval = '1h'
# 1) Build full_time_index once
all_times = pd.concat([
    bp['utcChartTime'],
    cate['utcChartTime'],
    intake['utcChartTime'],
    labs['utcChartTime'],
    sitecare['utcChartTime'],
    vitals_wide['utcChartTime']
]).dt.floor(sampling_interval)

full_time_index = pd.date_range(
    start=all_times.min(),
    end=all_times.max(),
    freq=sampling_interval
)


vitals_fast = fast_resample_and_reindex(
    vitals_wide,
    time_column='utcChartTime',
    interval=sampling_interval,
    full_time_index=full_time_index,
    const_cols=[]
)
del vitals_wide

# 2) Resample each table, watching the printed steps
bp_fast = fast_resample_and_reindex(
    bp,
    time_column='utcChartTime',
    interval=sampling_interval,
    full_time_index=full_time_index,
    const_cols=['age', 'sex_or_gender_numeric']
)
del bp

cate_fast = fast_resample_and_reindex(
    cate,
    time_column='utcChartTime',
    interval=sampling_interval,
    full_time_index=full_time_index,
    const_cols=[]
)
del cate
intake_fast = fast_resample_and_reindex(
    intake,
    time_column='utcChartTime',
    interval=sampling_interval,
    full_time_index=full_time_index,
    const_cols=[]
)
del intake
labs_fast = fast_resample_and_reindex(
    labs,
    time_column='utcChartTime',
    interval=sampling_interval,
    full_time_index=full_time_index,
    const_cols=[]
)
del labs
sitecare_fast = fast_resample_and_reindex(
    sitecare,
    time_column='utcChartTime',
    interval=sampling_interval,
    full_time_index=full_time_index,
    const_cols=[]
)
del sitecare





# 3) Merge them all—and watch prints to verify each DataFrame was built
merged_df = bp_fast
for df in [cate_fast, intake_fast, labs_fast, sitecare_fast, vitals_fast]:
    print("Merging another table into merged_df; current shape:", merged_df.shape)
    merged_df = merged_df.merge(
        df,
        on=['encounterId', 'utcChartTime'],
        how='outer'
    )
    print("   → New merged_df shape:", merged_df.shape)

# 4) Finally bring in demographics
print("Merging demographics; merged_df shape before:", merged_df.shape)
merged_df = merged_df.merge(demographics, on='encounterId', how='left')
print("Final merged_df shape:", merged_df.shape)

print("Done! Your merged DataFrame has", merged_df.shape, "rows ×", merged_df.shape[1], "columns.")


In [ ]:
merged_df.to_parquet('data/first_step_merged.parquet')

In [ ]:
merged_df
